In [1]:
import pandas as pd
import re

DATA_PATH = "S:/campaignfinance/data/"

# Race Level Data
The preprocessing code takes data on firm contributions, candidate information, and election ratings and output a complete firm-race level dataframe for 2010-2024

In [2]:
# Reading in files
contributions = pd.read_parquet(f"{DATA_PATH}FEC/processed/firm_candidate_contributions.parquet")
candidates = pd.read_csv(f"{DATA_PATH}/FEC/API/candidates/candidate_history_H.csv", dtype = {"address_zip": "str"})
elections = pd.read_csv(f"{DATA_PATH}/electionratings/IE/house_ratings.csv", index_col = 0)

In [4]:
# Process elections
elections["date"] = pd.to_datetime(elections["date"])
elections["year"] = elections["date"].dt.year

# Aggregate ratings by taking the mean, and interpreting this as continuous values from -4 to 4
elections = elections.groupby(by = ["state", "district", "year"]).agg(
    {"incumbent_party": "first",
     "incumbent": "first",
     "open": "first",
     "special": "first",
     "rating": "mean"}
).reset_index()

elections["flip"] = ((elections["incumbent_party"] == "republican") * (elections["rating"] < 0)).astype(int)
elections["uncertainty"] = abs(elections["rating"])

In [5]:
# Process candidates
candidates = candidates.dropna(subset = ["candidate_election_year", "district"])
candidates["year"] = candidates["candidate_election_year"].astype(int)
candidates["district"] = candidates["district"].astype(int)
vars = ["candidate_id", "state", "district", "year", "incumbent_challenge", "party"]
candidates = candidates[vars]
candidates = candidates.reset_index(drop = True)

In [6]:
# Process contributions
contributions = contributions[contributions["transaction_type"] == "24K"]
contributions = contributions[contributions["cand_id"].str.startswith("H")]
vars = ["cmte_id", "name", "amount", "cand_id", "cand_cmte", "year", "month", "day",
        "cmte_nm", "connected_org", "corporation", "instrument", "cusip",
        "hq", "hq_state", "hq_city", "ric", "permid", "GICS_Sector", "GICS_Industry_Group",
        "GICS_Industry", "GIS_Subindustry", "TRBC_Econ_Sector", "TRBC_Business_Sector",
        "TRBC_Industry_Group", "TRBC_Industry", "NAICS_Sector", "NAICS_Subsector",
        "NAICS_Industry_Group"]
contributions = contributions[vars]
contributions = contributions[contributions["month"].between(cutoff_start, cutoff_end, inclusive='both')]

In [7]:
# Merging all three data
merged_data = pd.merge(contributions, candidates,
                       how = "inner", left_on = ["cand_id", "year"],
                       right_on = ["candidate_id", "year"])
merged_date = merged_data.drop(columns = "cand_id")
merged_data = pd.merge(merged_data, elections,
                       how = "inner", on = ["state", "district", "year"])
merged_data.to_parquet(f"{DATA_PATH}house_all_data.parquet")

In [8]:
# Calculate inner products
products = (
    merged_data[merged_data["amount"] > 0]
    .groupby(by = ["cmte_id", "year", "state", "district"])
    .agg(
        total_amount = ("amount", "sum"),
        total_count = ("amount", "count"),
        nparty = ("party", lambda x: len(x.unique()))
    ).reset_index()
)
products["contribute"] = 1

In [ ]:
# Construct exhaustive firm-race pairings
vars = ["cmte_id", "year", "corporation", "instrument", "cusip",
        "hq", "hq_state", "hq_city", "ric", "permid", "GICS_Sector", "GICS_Industry_Group",
        "GICS_Industry", "GIS_Subindustry", "TRBC_Econ_Sector", "TRBC_Business_Sector",
        "TRBC_Industry_Group", "TRBC_Industry", "NAICS_Sector", "NAICS_Subsector",
        "NAICS_Industry_Group"]
corporate_pacs = contributions[vars].drop_duplicates()
corporate_pacs.to_parquet(f"{DATA_PATH}/FEC/processed/firm_pacs.parquet")

race_data = pd.merge(corporate_pacs, elections,
                     on = "year", how = "inner")

race_data = pd.merge(race_data, products,
                     on = ["cmte_id", "year", "state", "district"],
                     how = "left")

race_data["total_amount"] = race_data["total_amount"].fillna(0)
race_data["total_count"] = race_data["total_count"].fillna(0)
race_data["nparty"] = race_data["nparty"].fillna(0)
race_data["contribute"] = race_data["contribute"].fillna(0)

race_data.to_parquet(f"{DATA_PATH}house_firm_race.parquet")

# Candidate Level Data

In [2]:
merged_data = pd.read_parquet(f"{DATA_PATH}house_all_data.parquet")
general_cands = pd.read_csv(f"{DATA_PATH}other/house_general_cands.csv")
candidates = pd.read_csv(f"{DATA_PATH}/FEC/API/candidates/candidate_history_H.csv", dtype = {"address_zip": "str"})
corporate_pacs = pd.read_parquet(f"{DATA_PATH}/FEC/processed/firm_pacs.parquet")

In [3]:
def remove_punc(text):
    pattern = r'[^a-zA-Z0]'
    cleaned_text = re.sub(pattern, '', text)
    return cleaned_text

def parse_last_name(x):
    suffixes = {"JR", "SR", "II", "III", "IV", "V"}

    before_comma = x.split(",")[0].strip()
    parts = before_comma.split()
    while parts and parts[-1].upper().strip(".") in suffixes:
        parts.pop()
    if not parts:
        return ""
    last = parts[-1].split("-")[-1]
    last_clean = ''.join(c for c in last if c.isalpha())

    return last_clean

In [4]:
# Process candidates
candidates = candidates.dropna(subset = ["candidate_election_year", "district"])
candidates["year"] = candidates["candidate_election_year"].astype(int)
candidates["district"] = candidates["district"].astype(int)
vars = ["candidate_id", "candidate_last_name", "candidate_first_name", "name", "state", "district", "year", "incumbent_challenge", "party"]
candidates = candidates[vars]
candidates["last_name"] = candidates["name"].apply(parse_last_name)
candidates = candidates.reset_index(drop = True)

In [5]:
general_cands = general_cands[general_cands["year"] >= 2010]
general_cands = general_cands[~general_cands["writein"]]
general_cands = general_cands[~general_cands["party"].isna()]
general_cands = general_cands.reset_index(drop = True)
general_cands["last_name"] = general_cands["candidate"].apply(parse_last_name)
vars = ["year", "state_po", "district", "candidate", "party", "candidatevotes", "totalvotes", "last_name"]
general_cands = general_cands[vars]
general_cands = general_cands.rename(columns = {"state_po": "state"})

In [6]:
filter_cands = pd.merge(general_cands, candidates,
                        on = ["year", "state", "district", "last_name"],
                        how = "left")
filter_cands = filter_cands.rename(columns = {
    "party_x": "party",
    "party_y": "party_abb"
})
filter_cands = filter_cands.dropna(subset = "candidate_id")
vars = ["year", "state", "district", "candidate", "party", "candidatevotes", "totalvotes", "candidate_id", "incumbent_challenge"]
filter_cands = filter_cands[vars]

In [7]:
products = (
    merged_data[merged_data["amount"] > 0]
    .groupby(by = ["cmte_id", "year", "cand_id"])
    .agg(
        total_amount = ("amount", "sum"),
        total_count = ("amount", "count"),
        nparty = ("party", lambda x: len(x.unique()))
    ).reset_index()
    .rename(columns = {"cand_id": "candidate_id"})
)
products["contribute"] = 1

In [8]:
cand_data = pd.merge(corporate_pacs, filter_cands,
                     on = "year", how = "inner")
cand_data = pd.merge(cand_data, products,
                     on = ["cmte_id", "year", "candidate_id"],
                     how = "left")

In [ ]:
cand_data["total_amount"] = cand_data["total_amount"].fillna(0)
cand_data["total_count"] = cand_data["total_count"].fillna(0)
cand_data["nparty"] = cand_data["nparty"].fillna(0)
cand_data["contribute"] = cand_data["contribute"].fillna(0)

cand_data.to_parquet(f"{DATA_PATH}candidate_firm_race.parquet")